In [0]:
# Clean and validate the churn labels before joining them with the Customer 360 table. 
# Import Required Libraries 
from pyspark.sql.functions import *

In [0]:
# Load Bronze Churn Table 
churn = spark.table("churn_labels_100_k")

In [0]:
# View Data 
display(churn)

CustomerID,Churn
100000,0
100001,1
100002,0
100003,0
100004,0
100005,0
100006,1
100007,0
100008,0
100009,0


In [0]:
# View Schema 
churn.printSchema()

root
 |-- CustomerID: long (nullable = true)
 |-- Churn: long (nullable = true)



In [0]:
# churn.printSchema()

In [0]:
# Verify Record Count 
print("Total Records :", churn.count())

Total Records : 100000


In [0]:
# Check Missing Values 
churn.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in churn.columns
]).show()

+----------+-----+
|CustomerID|Churn|
+----------+-----+
|         0|    0|
+----------+-----+



In [0]:
# Remove Duplicate Customers. Every customer should have only one churn label. 
silver_churn = churn.dropDuplicates(["CustomerID"])

In [0]:
print("Original:", churn.count())

print("After Duplicate Removal:", silver_churn.count())

Original: 100000
After Duplicate Removal: 100000


In [0]:
# Handle Missing Labels, If Churn is missing
silver_churn = silver_churn.fillna({
    "Churn":0
})

In [0]:
#Convert Label Type , Ensure the label is integer. 
silver_churn = silver_churn.withColumn(
    "Churn",
    col("Churn").cast("integer")
)

In [0]:
silver_churn.printSchema()

root
 |-- CustomerID: long (nullable = true)
 |-- Churn: integer (nullable = false)



In [0]:
# Validate Labels 
silver_churn.filter(
    ~col("Churn").isin([0,1])
).show()

+----------+-----+
|CustomerID|Churn|
+----------+-----+
+----------+-----+



In [0]:
silver_churn = silver_churn.filter(
    col("Churn").isin([0,1])
)

In [0]:
# Create Churn Status 
silver_churn = silver_churn.withColumn(

    "ChurnStatus",

    when(col("Churn")==1,"CHURNED")

    .otherwise("ACTIVE")
)

In [0]:
# Create Retention Flag 
silver_churn = silver_churn.withColumn(

    "RetentionFlag",

    when(col("Churn")==0,1)

    .otherwise(0)
)

In [0]:
# Calculate Class Distribution 
silver_churn.groupBy("ChurnStatus").count().show()

+-----------+-----+
|ChurnStatus|count|
+-----------+-----+
|     ACTIVE|82040|
|    CHURNED|17960|
+-----------+-----+



In [0]:
# Calculate Churn Percentage 
total = silver_churn.count()

silver_churn.groupBy("ChurnStatus").count() \
.withColumn(
    "Percentage",
    round(col("count")*100/total,2)
).show()

+-----------+-----+----------+
|ChurnStatus|count|Percentage|
+-----------+-----+----------+
|     ACTIVE|82040|     82.04|
|    CHURNED|17960|     17.96|
+-----------+-----+----------+



In [0]:
# Verify Output 
display(silver_churn)

CustomerID,Churn,ChurnStatus,RetentionFlag
100000,0,ACTIVE,1
100001,1,CHURNED,0
100002,0,ACTIVE,1
100003,0,ACTIVE,1
100004,0,ACTIVE,1
100005,0,ACTIVE,1
100006,1,CHURNED,0
100007,0,ACTIVE,1
100008,0,ACTIVE,1
100009,0,ACTIVE,1


In [0]:
# Verify Schema 
silver_churn.printSchema()

root
 |-- CustomerID: long (nullable = true)
 |-- Churn: integer (nullable = false)
 |-- ChurnStatus: string (nullable = false)
 |-- RetentionFlag: integer (nullable = false)



In [0]:
# Save Silver Churn

# If this is the first time: 
silver_churn.write \
.mode("overwrite") \
.saveAsTable("silver_churn")

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-4678701858533686>, line 6
      1 # Save Silver Churn
      2 
      3 # If this is the first time: 
      4 silver_churn.write \
      5 .mode("overwrite") \
----> 6 .saveAsTable("silver_churn")

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/readwriter.py:737, in DataFrameWriter.saveAsTable(self, name, format, mode, partitionBy, **options)
    735 self._write.table_name = name
    736 self._write.table_save_method = "save_as_table"
--> 737 _, _, ei = self._spark.client.execute_command(
    738     self._write.command(self._spark.client), self._write.observations
    739 )
    740 self._callback(ei)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/client/core.py:1538, in SparkConnectClient.execute_command(self, command, observations, extra_request_metadata)
   1536     re

In [0]:
spark.sql("DROP TABLE IF EXISTS silver_churn")

DataFrame[]

In [0]:
silver_churn.write \
.mode("overwrite") \
.saveAsTable("silver_churn")

In [0]:
silver_churn.write \
.mode("overwrite") \
.option("overwriteSchema","true") \
.saveAsTable("silver_churn")

In [0]:
# Validate Saved Table 
display(spark.table("silver_churn"))

CustomerID,Churn,ChurnStatus,RetentionFlag
100030,0,ACTIVE,1
100036,0,ACTIVE,1
100042,0,ACTIVE,1
100043,0,ACTIVE,1
100060,0,ACTIVE,1
100068,0,ACTIVE,1
100082,0,ACTIVE,1
100084,0,ACTIVE,1
100135,0,ACTIVE,1
100140,1,CHURNED,0
